# Notebook 4: Out-of-Sample Backtest & Quant Performance Report

Trong notebook này, chúng ta thực hiện kiểm định (Backtest) chiến lược trên dữ liệu hoàn toàn chưa biết (Out-of-Sample từ `2023-07-01` đến `2024-06-30`):
1. Mô phỏng tái cơ cấu danh mục hàng tuần/hàng kỳ với chi phí giao dịch và thuế (`0.20%/trade`).
2. Tính toán bảng KPI tài chính định lượng chuyên nghiệp: **Sharpe Ratio**, **Maximum Drawdown**, **CAGR**, **Information Ratio**.
3. Trực quan hóa biểu đồ chuẩn xuất bản: Equity Curve, Underwater Drawdowns, Rolling Beta, và Allocation History.


In [1]:
!git clone https://github.com/PTN2004/AI-Driven-Market-Neutral-Portfolio-Optimization.git
%cd AI-Driven-Market-Neutral-Portfolio-Optimization
!pip install -qr requirements.txt

Cloning into 'AI-Driven-Market-Neutral-Portfolio-Optimization'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 101 (delta 0), reused 101 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 3.22 MiB | 32.64 MiB/s, done.
/content/AI-Driven-Market-Neutral-Portfolio-Optimization
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.7/278.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.8 MB/s eta 0:00:00


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import Config
from src.data_pipeline import DataFetcher, DataCleaner, FeatureEngineer, DataPreprocessor
from src.models import create_dataloaders, AlphaMLP, AlphaTrainer, AlphaPredictor
from src.risk_models import RiskModel, BetaCalculator
from src.optimization import PortfolioOptimizer
from src.backtest import BacktestEngine, PerformanceEvaluator, BacktestVisualizer

%matplotlib inline
sns.set_theme(style='whitegrid')


## 1. Chuẩn bị Pipeline và Mô hình AI đã huấn luyện


In [4]:
fetcher = DataFetcher()
raw_data = fetcher.fetch_all()
cleaned_data = DataCleaner().clean_and_align(raw_data)
feature_data = FeatureEngineer().compute_all_features(cleaned_data)
preprocessor = DataPreprocessor()
normalized_data = preprocessor.normalize_features(feature_data)

master_df = preprocessor.prepare_tabular_dataset(normalized_data, Config.START_DATE, Config.END_DATE)
train_loader, val_loader, test_loader, test_df = create_dataloaders(master_df, batch_size=64)

model = AlphaMLP(input_dim=len(FeatureEngineer.get_feature_names()))
trainer = AlphaTrainer(model, lr=1e-3)
history = trainer.fit(train_loader, val_loader, epochs=15)

predictor = AlphaPredictor(model)
test_df_with_preds = predictor.predict_all(test_df)
for sym in normalized_data.keys():
    if sym == Config.BENCHMARK_TICKER:
        continue
    sym_preds = test_df_with_preds[test_df_with_preds['symbol'] == sym].set_index('date')['predicted_mu']
    normalized_data[sym] = normalized_data[sym].merge(sym_preds, on='date', how='left')
    normalized_data[sym]['predicted_mu'] = normalized_data[sym]['predicted_mu'].ffill().fillna(0.0)


[2026-07-30 19:18:41] [INFO] [DataFetcher]: Starting data ingestion for 51 symbols (2021-01-01 -> 2024-06-30)...


INFO:DataFetcher:Starting data ingestion for 51 symbols (2021-01-01 -> 2024-06-30)...


[2026-07-30 19:18:41] [INFO] [DataFetcher]: Successfully loaded data for 51 symbols.


INFO:DataFetcher:Successfully loaded data for 51 symbols.


[2026-07-30 19:18:41] [INFO] [DataCleaner]: Aligning 50 symbols to benchmark (913 trading days)...


...

INFO:AlphaTrainer:Epoch 10/15 - Train Loss: 0.4688 - Val Loss: 0.5098 - Val IC: -0.0095


[2026-07-30 19:19:15] [INFO] [AlphaTrainer]: Epoch 15/15 - Train Loss: 0.4632 - Val Loss: 0.5059 - Val IC: 0.0009


INFO:AlphaTrainer:Epoch 15/15 - Train Loss: 0.4632 - Val Loss: 0.5059 - Val IC: 0.0009


[2026-07-30 19:19:15] [INFO] [AlphaTrainer]: Training completed.


INFO:AlphaTrainer:Training completed.


In [13]:
trainer.save_model("best_w.pt")

## 2. Thực thi Backtest Engine Out-of-Sample (Tái cơ cấu Hàng tuần)


In [5]:
engine = BacktestEngine(
    predictor=predictor,
    risk_model=RiskModel(),
    beta_calc=BetaCalculator(),
    optimizer=PortfolioOptimizer(),
    rebalance_freq='weekly',
    fee_rate=0.0020
)

backtest_res = engine.run(cleaned_data, normalized_data, start_date=Config.TEST_START_DATE, end_date=Config.END_DATE)
results_df = backtest_res['results_df']
weights_df = backtest_res['weights_df']
print('Hoàn tất Backtest Out-of-Sample!')


[2026-07-30 19:19:16] [INFO] [BacktestEngine]: Starting Out-of-Sample backtest (2023-07-01 -> 2024-06-30)...


INFO:BacktestEngine:Starting Out-of-Sample backtest (2023-07-01 -> 2024-06-30)...


[2026-07-30 19:19:41] [INFO] [BacktestEngine]: Out-of-Sample backtest completed successfully.


INFO:BacktestEngine:Out-of-Sample backtest completed successfully.


Hoàn tất Backtest Out-of-Sample!


## 3. Báo cáo Tóm tắt Hiệu suất (KPI Summary Table)


In [6]:
evaluator = PerformanceEvaluator()
summary_df = evaluator.evaluate(results_df)
display(summary_df)


[2026-07-30 19:19:41] [INFO] [PerformanceEvaluator]: Calculating quantitative performance metrics...


INFO:PerformanceEvaluator:Calculating quantitative performance metrics...


,AI Market Neutral,VN-Index (Buy & Hold),Equal-Weight VN100
Cumulative Return,35.18%,10.65%,14.20%
Annualized Return (CAGR),35.84%,10.83%,14.45%
Annualized Volatility,10.90%,17.03%,13.62%
Sharpe Ratio,2.59,0.51,0.84
Max Drawdown (MDD),-6.28%,-17.45%,-13.56%
Daily Win Rate,56.05%,58.47%,60.48%
Information Ratio,1.01,0.00,0.28


## 4. Biểu đồ Đường cong Tài sản & Quản trị Rủi ro


In [33]:
Config.DATA_DIR = "content/drive/MyDrive"
print(Config.DATA_DIR)

content/drive/MyDrive


In [36]:
visualizer = BacktestVisualizer(output_dir=Path('notebook_charts'))

# 1. Equity Curves
visualizer.plot_equity_curves(results_df)
plt.show()

# 2. Drawdowns
visualizer.plot_drawdowns(results_df)
plt.show()

# 3. Rolling Beta Verification
visualizer.plot_rolling_beta(results_df)
plt.show()

# 4. Long/Short Exposure
visualizer.plot_exposure_history(weights_df)
plt.show()
